### CoT ```<think></think><answer></answer>``` test

In [3]:
# from unsloth import FastVisionModel
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
import torch

In [2]:
base_model_name = "unsloth/Qwen3.5-4B"

In [3]:
adapter_path = "/home/ubuntu/Shree_FYP/data/outputs/stage_1/checkpoint-46000"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

In [5]:
base_model = AutoModelForImageTextToText.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [6]:
# Load LoRa adapter

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

In [7]:
# Merge models

model = model.merge_and_unload()

In [51]:
prompt = """I want you to think and give me the final answer in between <answer> and </answer> tags.
For example: <think> your reasoning goes here </think><answer>your final answer goes here</answer>
Do not put your <answer> and </answer> before </think>. Put it after
Question: I'm going to wash my car but its only 50 meters away, should I drive or walk?"""

In [52]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

In [53]:
input_text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
    return_dict = True,
    enable_thinking=True,
)

In [54]:
print(input_text)

<|im_start|>user
I want you to think and give me the final answer in between <answer> and </answer> tags.
For example: <think> your reasoning goes here </think><answer>your final answer goes here</answer>
Do not put your <answer> and </answer> before </think>. Put it after
Question: I'm going to wash my car but its only 50 meters away, should I drive or walk?<|im_end|>
<|im_start|>assistant
<think>



In [55]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
    enable_thinking=True,
).to(model.device)

In [56]:
outputs = model.generate(
    **inputs,
    max_new_tokens = 4096,
    temperature = 1.5,
    min_p = 0.1,
    use_cache = True,
)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


In [57]:
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

user
I want you to think and give me the final answer in between <answer> and </answer> tags.
For example: <think> your reasoning goes here </think><answer>your final answer goes here</answer>
Do not put your <answer> and </answer> before </think>. Put it after
Question: I'm going to wash my car but its only 50 meters away, should I drive or walk?
assistant
<think>
The user is asking whether they should drive or walk to wash their car, given that it's only 50 meters away.

Let me think about this:
- 50 meters is a very short distance
- Walking 50 meters takes about 30-45 seconds
- Driving 50 meters would require parking, getting in the car, starting the engine, etc.
- The effort of driving would be much greater than walking for such a short distance
- Walking is more practical and efficient for this short distance

The most logical answer is to walk, as it's more practical for such a short distance.
</think>

<answer>walk</answer>



### Verbalizer test - Qwen 3.5 4B latent student & Qwen 3.5 0.8B verbalizer

In [9]:
processor = AutoProcessor.from_pretrained("unsloth/Qwen3.5-0.8B")
model = AutoModelForImageTextToText.from_pretrained("unsloth/Qwen3.5-0.8B", device_map="cuda")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [10]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]

In [11]:
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

In [12]:
outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Based on the image, the animal on the candy is a **fish**.

You can see a black, stylized fish symbol painted on the surface of the green and orange candies. The fish is
